# OpenAI 도구 호출: JSON을 따라가는 실습

`사용자 질문 → 모델의 함수 호출 → Python 실행 → 결과 전달 → 최종 답변`을 관찰합니다.
실행하면 실제 API 사용량이 발생합니다. 프로젝트 루트의 `.env`를 사용합니다.

In [ ]:
from pathlib import Path
import sys

# Jupyter를 저장소 루트나 챕터 폴더에서 시작해도 모듈을 찾습니다.
base = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
example_dir = base / "notebooks/ch03/openai_tool_example"
sys.path.insert(0, str(example_dir))
from agent import TOOLS, DEFAULT_QUESTION, load_settings, run_agent
from openai import OpenAI
import json

model = load_settings()
print("모델:", model)  # API 키는 출력하지 않습니다.

## 1. 도구 정의

이 스키마가 API 요청의 `tools`에 들어갑니다. 모델에 Python 함수 코드를 보내는 것은 아닙니다.
`required`와 `additionalProperties: false`는 허용할 인자를 명시합니다.

In [ ]:
print(json.dumps(TOOLS, ensure_ascii=False, indent=2))

## 2. 에이전트 실행

각 request/response 전체 JSON을 출력합니다. `response.output`에서 `function_call`을 찾아보세요.
`arguments`는 JSON을 담은 문자열입니다.
다음 request의 `input`에서 `function_call_output.call_id`가 모델의 호출과 같은지 확인하세요.
다음 request의 `input`에는 이전 응답과 함수 실행 결과가 들어갑니다. 상세 반복 코드는 `agent.py`의 `run_agent`에 있습니다.

In [ ]:
question = DEFAULT_QUESTION
with OpenAI(timeout=60, max_retries=0) as client:
    answer, events = run_agent(client, question, model, example_dir / "log")

## 3. 기록 탐색

화면 출력과 같은 데이터를 `events` 및 `log/실행시각/` 아래 개별 JSON 파일에서 확인할 수 있습니다.
전체 응답에는 사용량 등 API가 반환한 필드가 포함됩니다. 모델 내부의 비공개 사고 과정은 제공되지 않습니다.

In [ ]:
print(answer)
print("기록 종류:", [e["kind"] for e in events])

# 첫 응답 전체 보기
first_response = next(e["data"] for e in events if e["kind"] == "response")
print(json.dumps(first_response, ensure_ascii=False, indent=2))

In [ ]:
# OpenAI에 보내는 요청 본문에서 도구 실행 결과 확인하기
for event in events:
    if event["kind"] == "request":
        print(json.dumps(event["data"], ensure_ascii=False, indent=2))

## 4. 바꿔보기

- `question`을 `안녕하세요`로 바꾸어 도구 없는 응답을 관찰하세요.
- `10을 0으로 나눠줘`로 바꾸어 오류가 도구 결과로 전달되는지 확인하세요.
- `TOOLS`와 `calculator` 함수에 연산을 추가해 보세요. 스키마와 구현을 함께 수정해야 합니다.

[공식 함수 호출 문서](https://developers.openai.com/api/docs/guides/function-calling)